In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

def churn_logistic_model(df, feature_cols, target_col):
    """
    Fit a logistic regression to predict churn.
    Standardize features first so coefficients are comparable.

    Parameters:
        df          : DataFrame with features and target
        feature_cols: list of feature column names
        target_col  : binary target column (1 = churned, 0 = retained)

    Returns a dict with:
        - feature_ranking : DataFrame with columns
                            ['feature', 'coefficient', 'odds_ratio']
                            sorted by abs(coefficient) descending
        - top_churn_driver: name of single most important feature
        - auc             : ROC-AUC on training data
        - n_obs           : number of observations used
        - churn_rate      : baseline churn rate in the dataset
    """
    # ── Input validation ──────────────────────────────────
    assert len(feature_cols) > 0, "feature_cols must not be empty"
    assert target_col in df.columns, f"{target_col} not in DataFrame"
    assert all(c in df.columns for c in feature_cols), "some feature_cols missing"
    assert df[target_col].nunique() == 2, "target must be binary (0/1)"

    # ── Step 1: Standardize features ─────────────────────
    # Fit on the data and transform — coefficients now in SD units
    # so they are directly comparable across features
    scaler = StandardScaler()
    X = scaler.fit_transform(df[feature_cols])   # shape: (n_obs, n_features)
    y = df[target_col].values

    # ── Step 2: Fit logistic regression ──────────────────
    logreg = LogisticRegression(max_iter=1000)
    logreg.fit(X, y)

    # ── Step 3: Extract coefficients + odds ratios ───────
    # coef_[0] because sklearn returns shape (1, n_features) for binary
    coefficients = logreg.coef_[0]
    odds_ratios  = np.exp(coefficients)

    # ── Step 4: Build ranking DataFrame ──────────────────
    # Sort by absolute coefficient so both positive and negative
    # drivers are ranked correctly
    feature_ranking = pd.DataFrame({
        'feature'    : feature_cols,
        'coefficient': coefficients,
        'odds_ratio' : odds_ratios
    }).sort_values('coefficient', key=abs, ascending=False)\
      .reset_index(drop=True)

    # ── Step 5: Remaining outputs ─────────────────────────
    top_churn_driver = feature_ranking.iloc[0]['feature']

    # predict_proba returns (n, 2) — column 1 = P(churn=1)
    auc = roc_auc_score(y, logreg.predict_proba(X)[:, 1])

    n_obs      = len(df)
    churn_rate = y.mean()

    # ── Output validation ─────────────────────────────────
    assert 0 <= auc <= 1
    assert len(feature_ranking) == len(feature_cols)

    return {
        'feature_ranking' : feature_ranking,
        'top_churn_driver': top_churn_driver,
        'auc'             : round(auc, 4),
        'n_obs'           : n_obs,
        'churn_rate'      : churn_rate
    }




In [3]:
# ── Sample data ───────────────────────────────────────────
np.random.seed(42)
n = 2000

df = pd.DataFrame({
    'sessions_day1_3'  : np.random.poisson(5, n),
    'avg_session_min'  : np.random.exponential(15, n),
    'friends_added'    : np.random.poisson(2, n),
    'levels_completed' : np.random.poisson(3, n),
    'spent_robux'      : np.random.exponential(50, n),
    'tutorial_complete': np.random.binomial(1, 0.6, n),
})

churn_score = (
    -0.8 * df['sessions_day1_3']
    -0.5 * df['tutorial_complete']
    -0.3 * df['friends_added']
    + np.random.normal(0, 1, n)
)
df['churned'] = (churn_score > churn_score.median()).astype(int)

feature_cols = ['sessions_day1_3', 'avg_session_min', 'friends_added',
                'levels_completed', 'spent_robux', 'tutorial_complete']

result = churn_logistic_model(df, feature_cols, target_col='churned')

print(f"AUC:              {result['auc']}")
print(f"N observations:   {result['n_obs']}")
print(f"Baseline churn:   {result['churn_rate']:.2%}")
print(f"Top churn driver: {result['top_churn_driver']}")
print(f"\nFeature Ranking:")
print(result['feature_ranking'].to_string(index=False))




AUC:              0.9301
N observations:   2000
Baseline churn:   50.00%
Top churn driver: sessions_day1_3

Feature Ranking:
          feature  coefficient  odds_ratio
  sessions_day1_3    -3.073388    0.046264
    friends_added    -0.914682    0.400644
tutorial_complete    -0.444235    0.641315
  avg_session_min    -0.113896    0.892350
      spent_robux    -0.043234    0.957687
 levels_completed    -0.032210    0.968304
